# Inferencia do modelo final de defasagem

Este notebook:
- carrega o pipeline final exportado;
- reconstrói as features da base de origem (PEDE2022, PEDE2023, PEDE2024);
- roda previsao para as transicoes 2022->2023 e 2023->2024;
- calcula metricas de assertividade possiveis (MAE, RMSE, R2);
- salva os outputs em CSV para analise.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
BASE_DIR = Path('/workspaces/Datathon-Machine-Learning-Engineering')
DATA_FILE = BASE_DIR / 'data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx'
ARTIFACTS_DIR = BASE_DIR / 'api/artifacts'

MODEL_PATH = ARTIFACTS_DIR / 'modelo_defasagem_pipeline.joblib'
SCHEMA_PATH = ARTIFACTS_DIR / 'modelo_defasagem_schema.json'

OUT_22_23 = ARTIFACTS_DIR / 'predicoes_modelo_final_2022_2023.csv'
OUT_23_24 = ARTIFACTS_DIR / 'predicoes_modelo_final_2023_2024.csv'
OUT_METRICS = ARTIFACTS_DIR / 'metricas_modelo_final_transicoes.csv'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

with SCHEMA_PATH.open('r', encoding='utf-8') as f:
    schema = json.load(f)

final_pipeline = joblib.load(MODEL_PATH)
required_columns = schema['required_columns']

print('Modelo carregado:', MODEL_PATH)
print('Schema carregado:', SCHEMA_PATH)
print('Modelo final:', schema.get('model_name'))
print('Colunas requeridas:', required_columns)

In [ ]:
base_2022 = pd.read_excel(DATA_FILE, sheet_name='PEDE2022')
base_2023 = pd.read_excel(DATA_FILE, sheet_name='PEDE2023')
base_2024 = pd.read_excel(DATA_FILE, sheet_name='PEDE2024')

def add_suffix_to_columns(df, suffix, exclude_cols=('RA',)):
    rename_map = {c: c if c in exclude_cols else f'{c}_{suffix}' for c in df.columns}
    return df.rename(columns=rename_map)

df_2022 = add_suffix_to_columns(base_2022, '2022')
df_2023 = add_suffix_to_columns(base_2023, '2023')
df_2024 = add_suffix_to_columns(base_2024, '2024')

FASE_MAP = {
    '0': 'ALFA',
    '1': 'FASE 1',
    '2': 'FASE 2',
    '3': 'FASE 3',
    '4': 'FASE 4',
    '5': 'FASE 5',
    '6': 'FASE 6',
    '7': 'FASE 7',
    '8': 'FASE 8',
    '9': 'FASE 9',
}

IDADE_IDEAL_POR_FASE = {
    'ALFA': 8,
    'FASE 1': 10,
    'FASE 2': 12,
    'FASE 3': 14,
    'FASE 4': 15,
    'FASE 5': 16,
    'FASE 6': 17,
    'FASE 7': 18,
    'FASE 8': 18,
    'FASE 9': 18,
}

def norm_fase(series):
    fase_base = series.astype('string').str.strip().str.upper()
    fase_digit = fase_base.str.extract(r'(\d)', expand=False)
    return fase_digit.map(FASE_MAP).fillna(fase_base)

def pick_col(candidates, cols):
    for c in candidates:
        if c in cols:
            return c
    return None

def maybe_to_numeric(series, min_parse_ratio=0.7):
    if pd.api.types.is_numeric_dtype(series):
        return series
    cleaned = (
        series.astype('string')
        .str.replace('%', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    parsed = pd.to_numeric(cleaned, errors='coerce')
    non_null = cleaned.notna().sum()
    if non_null == 0:
        return series
    if parsed.notna().sum() / non_null >= min_parse_ratio:
        return parsed
    return series

def extract_year_frame(df_year, year, max_cat_levels=50):
    cols = df_year.columns.tolist()

    col_fase = pick_col([f'Fase_{year}'], cols)
    col_idade = pick_col([f'Idade_{year}', f'Idade 22_{year}', f'Idade_{str(year)[-2:]}'], cols)
    col_inde = pick_col([f'INDE 2024_{year}', f'INDE 2023_{year}', f'INDE 23_{year}', f'INDE 22_{year}'], cols)
    col_por = pick_col([f'Por_{year}', f'Portug_{year}', f'Português_{year}'], cols)
    col_mat = pick_col([f'Mat_{year}', f'Matem_{year}'], cols)
    col_ing = pick_col([f'Ing_{year}', f'Inglês_{year}', f'Ingles_{year}'], cols)
    col_ieg = pick_col([f'IEG_{year}'], cols)
    col_ida = pick_col([f'IDA_{year}'], cols)

    required = {
        'fase': col_fase,
        'idade': col_idade,
        'inde': col_inde,
        'portugues': col_por,
        'matematica': col_mat,
        'ingles': col_ing,
        'ieg': col_ieg,
        'ida': col_ida,
    }
    miss = [k for k, v in required.items() if v is None]
    if miss:
        raise ValueError(f'Ano {year}: colunas nao encontradas -> {miss}')

    out = df_year[['RA', col_fase, col_idade, col_inde, col_por, col_mat, col_ing, col_ieg, col_ida]].copy()
    out.columns = ['RA', 'Fase', 'Idade', 'INDE', 'Portugues', 'Matematica', 'Ingles', 'IEG', 'IDA']
    out['Ano'] = year

    out['Fase_adj'] = norm_fase(out['Fase'])
    for c in ['Idade', 'INDE', 'Portugues', 'Matematica', 'Ingles', 'IEG', 'IDA']:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    out['idade_ideal'] = out['Fase_adj'].map(IDADE_IDEAL_POR_FASE)
    out = out.dropna(subset=['RA', 'Fase_adj', 'Idade', 'idade_ideal']).copy()

    used_cols = {col_fase, col_idade, col_inde, col_por, col_mat, col_ing, col_ieg, col_ida}
    extra_cols = [c for c in cols if c.endswith(f'_{year}') and c not in used_cols]
    extra = df_year[extra_cols].copy()

    rename_extra = {}
    suffix_len = len(str(year)) + 1
    for c in extra.columns:
        base_name = c[:-suffix_len]
        rename_extra[c] = f'feat_{base_name}'
    extra = extra.rename(columns=rename_extra)
    extra = extra.loc[:, ~extra.columns.duplicated()].copy()

    for c in extra.columns:
        extra[c] = maybe_to_numeric(extra[c])

    cat_keep = []
    num_keep = []
    for c in extra.columns:
        if pd.api.types.is_numeric_dtype(extra[c]):
            num_keep.append(c)
        else:
            nunique = extra[c].nunique(dropna=True)
            if 1 < nunique <= max_cat_levels:
                cat_keep.append(c)

    extra = extra[num_keep + cat_keep]
    out = pd.concat([out, extra], axis=1)
    return out

def zscore_group(series):
    std = series.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.mean()) / std

y2022 = extract_year_frame(df_2022, 2022)
y2023 = extract_year_frame(df_2023, 2023)
y2024 = extract_year_frame(df_2024, 2024)

long_df = pd.concat([y2022, y2023, y2024], ignore_index=True)
work = long_df.copy()

gap_base = (work['Idade'] - work['idade_ideal']).clip(lower=0)
work['gap_idade'] = np.where(work['Fase_adj'].isin(['FASE 8', 'FASE 9']), 0, gap_base)

work['media_notas'] = work[['Portugues', 'Matematica', 'Ingles']].mean(axis=1)
group_keys = ['Ano', 'Fase_adj']
work['z_notas_fase'] = work.groupby(group_keys)['media_notas'].transform(zscore_group)
work['z_ieg_fase'] = work.groupby(group_keys)['IEG'].transform(zscore_group)

work['score_idade'] = work['gap_idade']
work['score_notas'] = (-work['z_notas_fase']).clip(lower=0) * 2.0
work['score_engajamento'] = (-work['z_ieg_fase']).clip(lower=0) * 1.0

work['target_score'] = (
    work['score_idade'] + work['score_notas'] + work['score_engajamento']
).clip(0, 10)

print('Base reconstruida:', work.shape)
print('Anos disponiveis:', sorted(work['Ano'].dropna().unique().tolist()))

In [ ]:
def run_transition_prediction(df_work, ano_t, ano_t1, model, req_cols):
    base_t = df_work.loc[df_work['Ano'] == ano_t, ['RA'] + req_cols].copy()
    real_t1 = df_work.loc[df_work['Ano'] == ano_t1, ['RA', 'target_score']].copy()
    real_t1 = real_t1.rename(columns={'target_score': 'y_real'})

    ds = base_t.merge(real_t1, on='RA', how='inner')

    for c in req_cols:
        if c not in ds.columns:
            ds[c] = np.nan

    X_pred = ds[req_cols].copy()
    ds['y_pred'] = model.predict(X_pred)
    ds['erro_abs'] = (ds['y_real'] - ds['y_pred']).abs()
    ds['ano_origem'] = ano_t
    ds['ano_destino'] = ano_t1
    ds['transicao'] = f'{ano_t}->{ano_t1}'

    metrics = {
        'transicao': f'{ano_t}->{ano_t1}',
        'n_amostras': int(len(ds)),
        'MAE': float(mean_absolute_error(ds['y_real'], ds['y_pred'])) if len(ds) > 0 else np.nan,
        'RMSE': float(np.sqrt(mean_squared_error(ds['y_real'], ds['y_pred']))) if len(ds) > 0 else np.nan,
        'R2': float(r2_score(ds['y_real'], ds['y_pred'])) if len(ds) > 1 else np.nan,
    }

    ordered_cols = ['RA', 'transicao', 'ano_origem', 'ano_destino', 'y_real', 'y_pred', 'erro_abs'] + req_cols
    ds = ds[ordered_cols].sort_values('RA').reset_index(drop=True)
    return ds, metrics

pred_22_23, met_22_23 = run_transition_prediction(work, 2022, 2023, final_pipeline, required_columns)
pred_23_24, met_23_24 = run_transition_prediction(work, 2023, 2024, final_pipeline, required_columns)

pred_22_23.to_csv(OUT_22_23, index=False)
pred_23_24.to_csv(OUT_23_24, index=False)

metrics_df = pd.DataFrame([met_22_23, met_23_24])
metrics_df[['MAE', 'RMSE', 'R2']] = metrics_df[['MAE', 'RMSE', 'R2']].round(4)
metrics_df.to_csv(OUT_METRICS, index=False)

print('CSV salvo:', OUT_22_23)
print('CSV salvo:', OUT_23_24)
print('CSV salvo:', OUT_METRICS)
print('')
print('Metricas por transicao:')
print(metrics_df)

print('')
print('Preview 2022->2023:')
print(pred_22_23.head(10))

print('')
print('Preview 2023->2024:')
print(pred_23_24.head(10))